# SPML HW3: Model Extraction

In this notebook you'll explore model extraction.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import trange
import matplotlib.pyplot as plt


import torchvision
from torchvision import transforms, datasets, models

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Loading CIFAR100 (5 points)

Load the `CIFAR100` dataset. Make sure you resize the images to be `224x224` (same as the input size of resnet).

In [ ]:
# Load CIFAR-100 dataset

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

trainset_cifar100 = datasets.CIFAR100(
    root='./data', train=True, download=True, transform=transform
)

testset_cifar100 = datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform
)

trainloader_cifar100 = DataLoader(trainset_cifar100, batch_size=64, shuffle=True)
testloader_cifar100 = DataLoader(testset_cifar100, batch_size=64, shuffle=False)

100%|██████████| 169M/169M [00:03<00:00, 49.4MB/s]


# Pre-trained ResNet34 (10 points)

Load a pre-trained ResNet34 and train it on the `CIFAR100` dataset.

In [ ]:
# Load pretrained ResNet-34 model

resnet34 = models.resnet34(pretrained=True)
resnet34.fc = nn.Linear(resnet34.fc.in_features, 100)
resnet34 = resnet34.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 179MB/s]


In [ ]:
# Train the model on CIFAR100

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet34.parameters(), lr=1e-4)

epochs = 10
resnet34.train()

for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in trainloader_cifar100:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = resnet34(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(trainloader_cifar100):.4f}")


Epoch [1/10], Loss: 1.3908
Epoch [2/10], Loss: 0.5421
Epoch [3/10], Loss: 0.2896
Epoch [4/10], Loss: 0.1855
Epoch [5/10], Loss: 0.1364
Epoch [6/10], Loss: 0.1121
Epoch [7/10], Loss: 0.1010
Epoch [8/10], Loss: 0.0850
Epoch [9/10], Loss: 0.0771
Epoch [10/10], Loss: 0.0802


You might want to save this model to avoid retraining.

In [ ]:
torch.save(resnet34.state_dict(), "resnet34_cifar100.pth")

# Model Extraction (20 points)

Here we use knowledge distillation to extract models. If you are confused after the instructions take a look at the next section to understand what we are trying to do. The general steps in Knowledge Distillation are as follows:

1. Set the victim (teacher) to evaluation mode and the attacker (student) to training mode.
2. Use the victim to find the logits for each batch of inputs.
3. Predict the attackers output for the same batch of inputs.
4. Define and reduce the loss function over the difference between logits from the victim and attacker (use KL-Divergence, ...)
5. Repeat steps 2-4 for the number of epochs.

Feel free to check out [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531) to get a better sense of the process.

In [ ]:
def knowledge_distillation(victim_model, attacker_model, loader, optimizer, epochs, T):
    victim_model.eval()
    attacker_model.train()

    kd_loss = nn.KLDivLoss(reduction='batchmean')

    for epoch in range(epochs):
        total_loss = 0.0

        for images, _ in loader:
            images = images.to(device)

            with torch.no_grad():
                victim_logits = victim_model(images)

            attacker_logits = attacker_model(images)

            loss = kd_loss(
                torch.log_softmax(attacker_logits / T, dim=1),
                torch.softmax(victim_logits / T, dim=1)
            ) * (T * T)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}] KD Loss: {total_loss/len(loader):.4f}")


Can you explain how we should set the temperature? Why is this choice appropriate for model extraction?

higher temperature produces softer probability distributions that reveal class similarities in the teacher model and this provides richer supervision than hard labels and allows the attacker to better approximate the victim decision boundaries so using a moderate value like T=5 balances stability and information transfer

# Attack Transferability (20 points)

Implement attacks such as FGSM or PGD, you can use code from previous homeworks or readily available libraries.

In [ ]:
# Load or implement attacks

def fgsm_attack(model, images, labels, epsilon):
    images.requires_grad = True
    outputs = model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels)
    model.zero_grad()
    loss.backward()
    grad = images.grad.data
    perturbed_images = images + epsilon * grad.sign()
    return torch.clamp(perturbed_images, 0, 1)

Fill in the following function to attack a model and report the accuracy of the victim on the adversarial examples generated using the available model.

In [ ]:
def transferability_attack(model, victim, loader, attack):
    model.eval()
    victim.eval()

    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        adv_images = attack(model, images, labels)
        outputs = victim(adv_images)
        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    acc = 100 * correct / total
    print(f"Victim accuracy on transferred adversarial examples: {acc:.2f}%")
    return acc

# CIFAR10 (35 points)

## Loading and Exploration (5 points)

First load the `CIFAR10` dataset.

In [ ]:
# Load CIFAR-10 dataset

trainset_cifar10 = datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)

testset_cifar10 = datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

trainloader_cifar10 = DataLoader(trainset_cifar10, batch_size=64, shuffle=True)
testloader_cifar10 = DataLoader(testset_cifar10, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [00:03<00:00, 46.5MB/s]


Which classes from the `CIFAR10` dataset are present in `CIFAR100` classes?

In [2]:
# Check if classes are present in both datasets

cifar10_classes = set(trainset_cifar10.classes)
cifar100_classes = set(trainset_cifar100.classes)

overlap = cifar10_classes.intersection(cifar100_classes)
print("Overlapping classes:", overlap)


Overlapping classes: {'truck', 'ship', 'airplane', 'automobile', 'bird', 'cat', 'dog', 'horse', 'frog', 'deer'}


Now use the test dataset from `CIFAR10` to extract the model.

## Pre-trained ResNet18 (10 points)

Use the pre-trained ResNet18 dataset and extract the model using knowledge distillation.

In [ ]:
# Load pretrained ResNet-18 model
attacker_pretrained = models.resnet18(pretrained=True)
attacker_pretrained.fc = nn.Linear(attacker_pretrained.fc.in_features, 100)
attacker_pretrained = attacker_pretrained.to(device)

optimizer = optim.Adam(attacker_pretrained.parameters(), lr=1e-4)

# Extract the model
knowledge_distillation(
    victim_model=resnet34,
    attacker_model=attacker_pretrained,
    loader=trainloader_cifar10,
    optimizer=optimizer,
    epochs=10,
    T=5
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s]


Epoch [1/10] KD Loss: 1.4829
Epoch [2/10] KD Loss: 0.9049
Epoch [3/10] KD Loss: 0.7049
Epoch [4/10] KD Loss: 0.5766
Epoch [5/10] KD Loss: 0.4947
Epoch [6/10] KD Loss: 0.4344
Epoch [7/10] KD Loss: 0.3933
Epoch [8/10] KD Loss: 0.3570
Epoch [9/10] KD Loss: 0.3314
Epoch [10/10] KD Loss: 0.3091


What is the accuracy of the extracted model on the `CIFAR100` test set?

In [ ]:
# Report accuracy on CIFAR100
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100 * correct / total

acc_pretrained = evaluate(attacker_pretrained, testloader_cifar100)
print(f"Accuracy: {acc_pretrained:.2f}%")

Accuracy: 60.39%


## ResNet18 (10 points)

Repeat the pervious steps but without pre-training.

In [ ]:
# Load ResNet-18 model
attacker_scratch = models.resnet18(pretrained=False)
attacker_scratch.fc = nn.Linear(attacker_scratch.fc.in_features, 100)
attacker_scratch = attacker_scratch.to(device)

optimizer = optim.Adam(attacker_scratch.parameters(), lr=1e-4)

# Extract the model
knowledge_distillation(
    victim_model=resnet34,
    attacker_model=attacker_scratch,
    loader=trainloader_cifar10,
    optimizer=optimizer,
    epochs=10,
    T=5
)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/10] KD Loss: 2.4073
Epoch [2/10] KD Loss: 1.9482
Epoch [3/10] KD Loss: 1.7257
Epoch [4/10] KD Loss: 1.5601
Epoch [5/10] KD Loss: 1.4153
Epoch [6/10] KD Loss: 1.2830
Epoch [7/10] KD Loss: 1.1482
Epoch [8/10] KD Loss: 1.0246
Epoch [9/10] KD Loss: 0.9157
Epoch [10/10] KD Loss: 0.8173


Measure the accuracy of the newly distillied attacker and compare your results from the previous section.

In [ ]:
# Report accuracy on CIFAR100

acc_scratch = evaluate(attacker_scratch, testloader_cifar100)
print(f"Accuracy: {acc_scratch:.2f}%")

Accuracy: 21.16%


What are the effects of pre-training?

pre training gives the attacker strong feature representations and making it easier to mimic the victim during distillation so the pre trained model converges faster and achieves higher accuracy than a model trained from scratch

## Full Dataset (10 points)

Repeat your experiments using the pre-trained ResNet18 but this time use the entire CIFAR10 dataset.

In [ ]:
# Load pretrained ResNet-18 model
attacker_full = models.resnet18(pretrained=True)
attacker_full.fc = nn.Linear(attacker_full.fc.in_features, 100)
attacker_full = attacker_full.to(device)

optimizer = optim.Adam(attacker_full.parameters(), lr=1e-4)

# Extract the model
knowledge_distillation(
    victim_model=resnet34,
    attacker_model=attacker_full,
    loader=trainloader_cifar10,
    optimizer=optimizer,
    epochs=15,
    T=5
)

Epoch [1/15] KD Loss: 1.4869
Epoch [2/15] KD Loss: 0.9077
Epoch [3/15] KD Loss: 0.7004
Epoch [4/15] KD Loss: 0.5752
Epoch [5/15] KD Loss: 0.4922
Epoch [6/15] KD Loss: 0.4337
Epoch [7/15] KD Loss: 0.3916
Epoch [8/15] KD Loss: 0.3574
Epoch [9/15] KD Loss: 0.3322
Epoch [10/15] KD Loss: 0.3076
Epoch [11/15] KD Loss: 0.2901
Epoch [12/15] KD Loss: 0.2741
Epoch [13/15] KD Loss: 0.2600
Epoch [14/15] KD Loss: 0.2465
Epoch [15/15] KD Loss: 0.2368


Report the accuracy on the `CIFAR100` testset once more.

In [4]:
# Report accuracy on CIFAR100
acc_full = evaluate(attacker_full, testloader_cifar100)
print(f"Accuracy: {acc_full:.2f}%")

Accuracy: 48.07%


What are the effects of using more data?

using more data improves model extraction by providing better coverage of the input space and this allows the attacker to learn a more accurate approximation of the victim behavior

# CIFAR100 (10 points)

This time, use the training dataset from `CIFAR100` and perform knowledge distillation on a pre-trained ResNet18.

In [ ]:
# Load pretrained ResNet-18 model
attacker_cifar100 = models.resnet18(pretrained=True)
attacker_cifar100.fc = nn.Linear(attacker_cifar100.fc.in_features, 100)
attacker_cifar100 = attacker_cifar100.to(device)

optimizer = optim.Adam(attacker_cifar100.parameters(), lr=1e-4)

# Extract the model
knowledge_distillation(
    victim_model=resnet34,
    attacker_model=attacker_cifar100,
    loader=trainloader_cifar100,
    optimizer=optimizer,
    epochs=10,
    T=5
)

Epoch [1/10] KD Loss: 5.2633
Epoch [2/10] KD Loss: 2.4181
Epoch [3/10] KD Loss: 1.6264
Epoch [4/10] KD Loss: 1.2055
Epoch [5/10] KD Loss: 0.9574
Epoch [6/10] KD Loss: 0.8148
Epoch [7/10] KD Loss: 0.7090
Epoch [8/10] KD Loss: 0.6395
Epoch [9/10] KD Loss: 0.5971
Epoch [10/10] KD Loss: 0.5544


How does the accuracy change now?

In [5]:
# Report accuracy on CIFAR100
acc_cifar100 = evaluate(attacker_cifar100, testloader_cifar100)
print(f"Accuracy: {acc_cifar100:.2f}%")

Accuracy: 65.91%


Why do you suppose using the `CIFAR100` had the following results? Explain your observations.

CIFAR100 matches the victim training distribution and eliminating domain mismatch so this allows the attacker to closely replicate the victim model and achieve much higher accuracy